## 데이터 로드

In [20]:
import pandas as pd 
df = pd.read_csv("./data/ending_club_preprocessed2.csv", parse_dates=['issue_d', 'earliest_cr_line'])
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import platform
warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
print("=" * 60)
print("로드 완료")
print("=" * 60)

로드 완료


## 분석을 위한 2차 피쳐 선택 

In [21]:
# 날자형# 대출 실행일 월별 변수 만들기 
df["issue_month"] = df["issue_d"].dt.month 

use_col=[
    "target",
    # 대출 기본정보 
    "loan_amnt", # 대출 금액 
    "term", # 기간
    "int_rate", #이자율 
    "sub_grade", # 등급세분류 , 
    "issue_month", # 월별 변수
    "installment", #월 상환액 ($)
    "purpose", # 목적 
    # 차입자 정보 
    "emp_length", # 근속연수 
    "home_ownership", # 주거 형태
    "annual_inc", # 자기보고 연소득 ($)
    # 부채 
    "dti", # 부채대비 소득 
    # 기타 사전 분석 기반 
    "pub_rec", # 부정기록 
    "initial_list_status", # 투자자 측 펀딩 방법 
    "addr_state", # 주 
    "earliest_cr_line",
    "avg_cur_bal",
    "acc_open_past_24mths",
    "fico_mid", 
    "bc_open_to_buy", 
    "revol_bal", 
    "num_sats",
    "pub_rec_bankruptcies", 
    "tax_liens", 
    # 계좌 경과 기간
    "mo_sin_old_il_acct", # 
    "mths_since_rcnt_il", 
    "mo_sin_old_rev_tl_op",
    "mo_sin_rcnt_rev_tl_op", 
    "mths_since_recent_inq", 
    "mths_since_recent_bc", 
    # 플래그
    "mths_since_rcnt_il_flag",
    "mths_since_recent_inq_flag",
    "mths_since_recent_bc_flag",
]

df = df[use_col].copy()
print(f"전체: {len(use_col)}")
print(f"고유: {len(set(use_col))}")

전체: 33
고유: 33


## 전처리 

In [22]:
# 전체 컬럼의 null 여부 
null_info = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측률': df.isnull().mean().apply(lambda x: f"{x:.2%}")
})
print(null_info[null_info['결측수'] > 0])

                          결측수     결측률
dti                       414   0.03%
mo_sin_old_il_acct      38045   2.98%
mths_since_rcnt_il     754388  59.04%
mths_since_recent_inq  124121   9.71%
mths_since_recent_bc    12685   0.99%


## Train/ Test 분리 

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

In [24]:
#  분리
y= df["target"] # target
X = df.drop(columns=["target"]) # 설명 변수 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 컬럼 구분 

num_cols  = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"수치형: {len(num_cols)}개\n{num_cols}\n")
print(f"범주형: {len(cat_cols)}개\n{cat_cols}")

수치형: 21개
['loan_amnt', 'int_rate', 'issue_month', 'installment', 'annual_inc', 'dti', 'pub_rec', 'avg_cur_bal', 'acc_open_past_24mths', 'fico_mid', 'bc_open_to_buy', 'revol_bal', 'num_sats', 'pub_rec_bankruptcies', 'tax_liens', 'mo_sin_old_il_acct', 'mths_since_rcnt_il', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mths_since_recent_inq', 'mths_since_recent_bc']

범주형: 10개
['term', 'sub_grade', 'purpose', 'emp_length', 'home_ownership', 'initial_list_status', 'addr_state', 'mths_since_rcnt_il_flag', 'mths_since_recent_inq_flag', 'mths_since_recent_bc_flag']


In [25]:
# 결측 처리 
# test data로 결측 처리 수치형 
# dti 
X_train["dti"] = X_train.groupby("sub_grade")["dti"].transform(
    lambda x: x.fillna(x.median())
)

train_dit_medi = X_train.groupby("sub_grade")["dti"].median()

X_test["dti"] = X_test["dti"].fillna(X_test["sub_grade"].map(train_dit_medi))

X_train= X_train.drop(columns=["sub_grade"])
X_test= X_test.drop(columns=["sub_grade"])

# 신용 조회, 
# 최대값 +1 
cols = [
'mths_since_recent_inq',
'mths_since_recent_bc', 
]
for col in cols:
    max_val = X_train[col].max()
    X_train[col] = X_train[col].fillna(max_val + 1) 
    X_test[col] = X_test[col].fillna(max_val + 1) 


## mo_sin_old_il_acct  가장 오래된 할부 계좌 이후 경과 월수
## 0 으로 
X_train["mo_sin_old_il_acct"] = X_train["mo_sin_old_il_acct"].fillna(0) 
X_test["mo_sin_old_il_acct"] =  X_test["mo_sin_old_il_acct"].fillna(0) 

# mths_since_rcnt_il 가장 최근 할부 계좌(Installment Loan) 개설 이후 경과 월수
## 중앙값으로 채우기 
medi_dlq = X_train["mths_since_rcnt_il"].median()
X_train["mths_since_rcnt_il"] = X_train["mths_since_rcnt_il"].fillna(medi_dlq)
X_test["mths_since_rcnt_il"] = X_test["mths_since_rcnt_il"].fillna(medi_dlq)


In [26]:
# 확인
null_train = X_train.isnull().sum()
print(null_train[null_train>0])

null_test = X_test.isnull().sum()
print(null_test[null_test>0])

Series([], dtype: int64)
Series([], dtype: int64)


## 데이터 증강 기법 
- CTGAN 사용 

In [28]:
# 설치 확인 
import sdv, ctgan
print(sdv.__version__, ctgan.__version__)
from sdv.single_table import CTGANSynthesizer  # SDV 1.x+ 경로

1.32.1 0.11.1


## 모델 생성